# Synthetic Data Generator
### Step 8 : Kafka Consumer Adapter

In [ ]:
from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.kafka_consumer_adapter import (
    run_kafka_consumer_to_postgres_once, 
    run_kafka_consumer_to_postgres_loop,
)

In [ ]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

KAFKA_TOPIC = str("pump.telemetry.synthetic")
CONSUMER_DESTINATION_TABLE_NAME = str("synethic_sensor_messages_consumed_stage")
CONSUMER_GROUP_ID = str("synthetic-telemetry-consumer-group")
CONSUMER_WORKER_ID = str("consumer_worker_001")

CONSUMER_MAX_MESSAGES = int(520)
CONSUMER_POLL_TIMEOUT_SECONDS = float(1.0)

AUTO_OFFSET_RESET = str("earliest")


MAX_BATCHES = int(10)
IDLE_SLEEP_SECONDS = float(0.0)
STOP_ON_EMPTY_FLAG = True


---

In [ ]:
engine = get_engine_from_env()

---

#### Single Batch

In [ ]:

result = run_kafka_consumer_to_postgres_once(
    engine=engine,
    topic=KAFKA_TOPIC,
    schema=SCHEMA,
    table_name=CONSUMER_DESTINATION_TABLE_NAME,
    max_messages=CONSUMER_MAX_MESSAGES,
    poll_timeout_seconds=CONSUMER_POLL_TIMEOUT_SECONDS,
    consumer_group_id=CONSUMER_GROUP_ID,
    consumer_worker_id=CONSUMER_WORKER_ID,
    auto_offset_reset=AUTO_OFFSET_RESET,
)

display(result)

----

#### Loop Batches

In [ ]:
results = run_kafka_consumer_to_postgres_loop(
    engine=engine,
    topic=KAFKA_TOPIC,
    schema=SCHEMA,
    table_name=CONSUMER_DESTINATION_TABLE_NAME,
    max_messages=CONSUMER_MAX_MESSAGES,
    poll_timeout_seconds=CONSUMER_POLL_TIMEOUT_SECONDS,
    consumer_group_id=CONSUMER_GROUP_ID,
    consumer_worker_id=CONSUMER_WORKER_ID,
    auto_offset_reset=AUTO_OFFSET_RESET,
    max_batches=MAX_BATCHES,
    idle_sleep_seconds=IDLE_SLEEP_SECONDS,
    stop_on_empty=STOP_ON_EMPTY_FLAG,
)

display(results)

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT kafka_topic || ':' || kafka_partition || ':' || kafka_offset) AS distinct_kafka_messages,
        MIN(consumer_received_at) AS min_consumer_received_at,
        MAX(consumer_received_at) AS max_consumer_received_at
    FROM capstone.synthetic_sensor_messages_consumed_stage
    """
)

display(validation_dataframe)

----

In [ ]:
# Dispose Engine
engine.dispose()